<a href="https://colab.research.google.com/github/GauthamAB/AI/blob/main/a_algo_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import heapq
import copy

class PuzzleNode:
    def __init__(self, state, parent=None, action=None, g=0, h=0):
        self.state = state  # 2D tuple representation of the board
        self.parent = parent
        self.action = action
        self.g = g  # Path cost from start to current node
        self.h = h  # Heuristic estimate to target
        self.f = g + h  # Estimated total cost

    def __lt__(self, other):
        return self.f < other.f

    def __eq__(self, other):
        return self.state == other.state

    def __hash__(self):
        return hash(self.state)

# Helper to find the blank tile (0)
def find_blank(state):
    for r in range(3):
        for c in range(3):
            if state[r][c] == 0:
                return r, c

# Get next legal states
def get_neighbors(state):
    neighbors = []
    r, c = find_blank(state)
    moves = {
        'Up': (-1, 0),
        'Down': (1, 0),
        'Left': (0, -1),
        'Right': (0, 1)
    }

    for action, (dr, dc) in moves.items():
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            # Create a mutable copy to swap tiles
            new_board = [list(row) for row in state]
            new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]
            neighbors.append((tuple(tuple(row) for row in new_board), action))
    return neighbors
# 1. Misplaced Tiles Heuristic
def h_misplaced_tiles(state, goal_state):
    misplaced = 0
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0 and val != goal_state[r][c]:
                misplaced += 1
    return misplaced

# 2. Manhattan Distance Heuristic
def h_manhattan_distance(state, goal_state):
    # Precompute goal coordinates for fast lookup
    goal_coords = {}
    for r in range(3):
        for c in range(3):
            goal_coords[goal_state[r][c]] = (r, c)

    distance = 0
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0:
                target_r, target_c = goal_coords[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance
def solve_a_star(start_state, goal_state, heuristic_fn):
    start_node = PuzzleNode(start_state, g=0, h=heuristic_fn(start_state, goal_state))

    open_set = []
    heapq.heappush(open_set, start_node)

    closed_set = set()
    nodes_expanded = 0

    while open_set:
        current_node = heapq.heappop(open_set)
        nodes_expanded += 1

        if current_node.state == goal_state:
            # Reconstruct path
            path = []
            curr = current_node
            while curr.parent is not None:
                path.append((curr.action, curr.state))
                curr = curr.parent
            path.reverse()
            return path, nodes_expanded

        closed_set.add(current_node.state)

        for neighbor_state, action in get_neighbors(current_node.state):
            if neighbor_state in closed_set:
                continue

            g_score = current_node.g + 1
            h_score = heuristic_fn(neighbor_state, goal_state)
            neighbor_node = PuzzleNode(neighbor_state, current_node, action, g_score, h_score)

            # Simple open_set check / push
            heapq.heappush(open_set, neighbor_node)

    return None, nodes_expanded
def print_board(state):
    for row in state:
        print(" ".join(str(x) if x != 0 else "_" for x in row))
    print()

# Define Start and Goal States (0 represents the empty space)
start_state = (
    (1, 2, 3),
    (0, 4, 6),
    (7, 5, 8)
)

goal_state = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 0)
)

print("--- Start State ---")
print_board(start_state)

print("--- Goal State ---")
print_board(goal_state)

# Solve with Misplaced Tiles
path_misplaced, expanded_misplaced = solve_a_star(start_state, goal_state, h_misplaced_tiles)
print(f"Misplaced Tiles Heuristic: Solved in {len(path_misplaced)} steps. Expanded {expanded_misplaced} nodes.")

# Solve with Manhattan Distance
path_manhattan, expanded_manhattan = solve_a_star(start_state, goal_state, h_manhattan_distance)
print(f"Manhattan Distance Heuristic: Solved in {len(path_manhattan)} steps. Expanded {expanded_manhattan} nodes.")

# Print Step-by-Step Solution
print("\n--- Step-by-Step Solution (Manhattan Distance) ---")
for step, (action, state) in enumerate(path_manhattan, 1):
    print(f"Step {step}: Move Blank {action}")
    print_board(state)

--- Start State ---
1 2 3
_ 4 6
7 5 8

--- Goal State ---
1 2 3
4 5 6
7 8 _

Misplaced Tiles Heuristic: Solved in 3 steps. Expanded 4 nodes.
Manhattan Distance Heuristic: Solved in 3 steps. Expanded 4 nodes.

--- Step-by-Step Solution (Manhattan Distance) ---
Step 1: Move Blank Right
1 2 3
4 _ 6
7 5 8

Step 2: Move Blank Down
1 2 3
4 5 6
7 _ 8

Step 3: Move Blank Right
1 2 3
4 5 6
7 8 _

